In [3]:
import pandas as pd
from collections import defaultdict
from google.colab import files

# ----------------- Step 1: Load dataset (Upload) -----------------
def load_dataset():
    return pd.read_csv("data1.csv")

# Load the dataset
df = load_dataset()

# Separate features and target
features = df.columns[:-1]
target = df.columns[-1]

# ----------------- Step 2: Build Frequency Table from Train -----------------
class_counts = df[target].value_counts().to_dict()
total_samples = len(df)

# Prior probabilities
priors = {cls: count / total_samples for cls, count in class_counts.items()}

# Frequency table
freq_table = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))

for _, row in df.iterrows():
    cls = row[target]
    for feature in features:
        value = row[feature]
        freq_table[feature][value][cls] += 1

print("\n=== Frequency Table (Train Data) ===")
for feature in features:
    print(f"\nFeature: {feature}")
    for value in freq_table[feature]:
        print(f" {value}: {dict(freq_table[feature][value])}")

# ----------------- Step 3: Likelihood Probabilities with Laplace Smoothing -----------------
likelihoods = defaultdict(lambda: defaultdict(dict))

for feature in features:
    values = df[feature].unique() # all possible values of this feature
    for value in values:
        for cls in class_counts:
            count = freq_table[feature][value][cls]
            likelihoods[feature][value][cls] = (count + 1) / (class_counts[cls] + len(values))

print("\n=== Likelihood Probabilities (Train Data, with Laplace smoothing) ===")
for feature in features:
    print(f"\nFeature: {feature}")
    for value in likelihoods[feature]:
        print(f" {value}: {likelihoods[feature][value]}")

# ----------------- Step 4: Naive Bayes Classification -----------------
def classify(sample):
    probs = {}
    for cls in class_counts:
        prob = priors[cls] # prior
        for feature in features:
            value = sample.get(feature, None)
            if value in likelihoods[feature]:
                prob *= likelihoods[feature][value][cls]
            else:
                # if completely unseen value → apply uniform smoothing
                prob *= 1 / (class_counts[cls] + len(df[feature].unique()))
        probs[cls] = prob

    # Normalize → posterior
    total = sum(probs.values())
    if total > 0:
        for cls in probs:
            probs[cls] /= total
    predicted = max(probs, key=probs.get)
    return predicted, probs

# ----------------- Step 5a: Test Using User Input -----------------
print("\nEnter values for a new sample:")
sample = {}
for feature in features:
    val = input(f"{feature}: ")
    sample[feature] = val

predicted, probs = classify(sample)

print("\n=== Result (User Input Sample) ===")
print("Input Sample:", sample)
print("Predicted Class:", predicted)
print("Posterior Probabilities:", probs)

# ----------------- Step 5b: Test Accuracy on Whole Dataset -----------------
correct = 0
for _, row in df.iterrows():
    sample = row.to_dict()
    true_class = sample[target]
    del sample[target]
    predicted, _ = classify(sample)
    if predicted == true_class:
        correct += 1

accuracy = correct / len(df)
print("\n=== Training Accuracy on CSV Data ===")
print("Accuracy:", accuracy)


=== Frequency Table (Train Data) ===

Feature: Outlook
 Sunny: {'No': 3, 'Yes': 2}
 Overcast: {'Yes': 4}
 Rainy: {'Yes': 3, 'No': 2}

Feature: Temperature
 Hot: {'No': 2, 'Yes': 2}
 Mild: {'Yes': 4, 'No': 2}
 Cool: {'Yes': 3, 'No': 1}

Feature: Humidity
 High: {'No': 4, 'Yes': 3}
 Normal: {'Yes': 6, 'No': 1}

Feature: Windy
 False: {'No': 2, 'Yes': 6}
 True: {'No': 3, 'Yes': 3}

=== Likelihood Probabilities (Train Data, with Laplace smoothing) ===

Feature: Outlook
 Sunny: {'Yes': 0.25, 'No': 0.5}
 Overcast: {'Yes': 0.4166666666666667, 'No': 0.125}
 Rainy: {'Yes': 0.3333333333333333, 'No': 0.375}

Feature: Temperature
 Hot: {'Yes': 0.25, 'No': 0.375}
 Mild: {'Yes': 0.4166666666666667, 'No': 0.375}
 Cool: {'Yes': 0.3333333333333333, 'No': 0.25}

Feature: Humidity
 High: {'Yes': 0.36363636363636365, 'No': 0.7142857142857143}
 Normal: {'Yes': 0.6363636363636364, 'No': 0.2857142857142857}

Feature: Windy
 False: {'Yes': 0.6363636363636364, 'No': 0.42857142857142855}
 True: {'Yes': 0.36363